# Tratando campos extras

Vamos começar a ver as opções de configuração de modelo pela forma como o Pydantic trata campos que estão nos dados desserializados
mas **não** estão definidos no modelo. São os campos "extras".

Existem muitas opções de configuração; a lista completa está [aqui](https://docs.pydantic.dev/latest/api/config).

In [1]:
from pydantic import BaseModel, ConfigDict, ValidationError

In [2]:
class Model(BaseModel):
    field_1: int

In [3]:
m = Model(field_1=10, field_2=20)
print(f"m: {m!r}")

m: Model(field_1=10)


`field_2` não aparece na representação e, de fato, também não está no dicionário da instância:

In [4]:
print(f"dict(m): {dict(m)!r}")

dict(m): {'field_1': 10}


O Pydantic oferece uma forma de ver quais campos vieram nos dados mas não estão definidos no modelo:

In [5]:
print(f"m.model_extra: {m.model_extra!r}")

m.model_extra: None


Até isso está vazio (`None`).

Ou seja, o padrão do Pydantic é **ignorar totalmente** campos extras.

Podemos mudar esse comportamento configurando o modelo.

A opção de configuração que procuramos se chama `extra`.

A documentação específica está na página acima, ou diretamente [aqui](https://docs.pydantic.dev/latest/api/config/#pydantic.config.ConfigDict.extra).

Pela documentação, `extra` aceita três valores:
- `ignore` — o comportamento padrão
- `forbid` — gera erro de validação se houver campos extras nos dados
- `allow` — adiciona o campo e seu valor ao modelo, mas, claro, sem validação.

Vejamos como definir explicitamente o comportamento `ignore` (é o padrão, então não precisaríamos — só para ilustrar).

In [6]:
class Model(BaseModel):
    model_config = ConfigDict(extra="ignore")

    field_1: int = 0

In [7]:
print(f'Model(field_1=10, extra_1="data"): {Model(field_1=10, extra_1="data")!r}')

Model(field_1=10, extra_1="data"): Model(field_1=10)


Agora vamos mudar para levantar exceção de validação quando houver campos extras.

In [8]:
class Model(BaseModel):
    model_config = ConfigDict(extra="forbid")

    field_1: int = 0

In [9]:
try:
    Model(field_1=10, extra_1="data")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
extra_1
  Extra inputs are not permitted [type=extra_forbidden, input_value='data', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


Agora recebemos uma exceção de validação. Isso é típico em APIs REST em que queremos garantir que o payload não traga dados a mais:
é melhor ser rigoroso e avisar o cliente de que há algo errado na requisição, já que ele pode nem perceber o problema do lado dele.

A última opção é permitir (e tratar) campos extras:

In [10]:
class Model(BaseModel):
    model_config = ConfigDict(extra="allow")

    field_1: int = 0

Agora podemos mandar campos adicionais (não definidos) e o Pydantic os adiciona à **instância** (nada de schema comum aqui: o schema
do **modelo** continua o mesmo).

In [11]:
m = Model(field_1=10, extra_1="data")
print(f"m: {m!r}")

m: Model(field_1=10, extra_1='data')


O campo ad-hoc aparece na representação e no dicionário da instância:

In [12]:
print(f"dict(m): {dict(m)!r}")

dict(m): {'field_1': 10, 'extra_1': 'data'}


Os campos do modelo não mudaram (lembre: `model_fields` se acessa pela classe):

In [13]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'field_1': FieldInfo(annotation=int, required=False, default=0)}


Ao serializar a instância, o dado extra aparece:

In [14]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'field_1': 10, 'extra_1': 'data'}


Podemos inspecionar como vimos numa aula anterior.

In [15]:
print(f"m.model_fields_set: {m.model_fields_set!r}")

m.model_fields_set: {'extra_1', 'field_1'}


O nome `extra_1` aparece nessa visão porque ele **foi** definido na instância, e a partir dos dados (não de um default).

Também há uma propriedade que lista só os campos extras:

In [16]:
print(f"m.model_extra: {m.model_extra!r}")

m.model_extra: {'extra_1': 'data'}


Recebemos um dicionário apenas com os campos extras — pode ser útil.

Por exemplo: você consulta uma API que devolve um JSON complexo, mas só lhe interessam algumas informações; o resto precisa ser guardado
para outro fim (talvez log). Em vez de modelar o JSON inteiro, você define só as partes necessárias, configura `allow` e valida apenas o
que interessa, levando o resto como está. Isso também reduz o trabalho de validação: não valide o que você não vai usar.

Uma última informação sobre configuração: sempre dá para recuperar a configuração de um modelo (ou de uma instância), o que costuma
ajudar na depuração:

In [17]:
print(f"Model.model_config: {Model.model_config!r}")

Model.model_config: {'extra': 'allow'}


In [18]:
print(f"m.model_config: {m.model_config!r}")

m.model_config: {'extra': 'allow'}


Como se vê, o atributo `model_config` é um simples `dict` (`ConfigDict` é um `TypedDict`, que em tempo de execução é um dict comum).

Dá, sim, para configurar o modelo com um dicionário comum:

In [19]:
class Model(BaseModel):
    model_config = {"extra": "allow"}

    field_1: int = 0

In [20]:
print(f"Model(extra_1=100): {Model(extra_1=100)!r}")

Model(extra_1=100): Model(field_1=0, extra_1=100)


O motivo para **não** fazer isso: `ConfigDict` é um `TypedDict` e oferece alguma proteção contra erros de configuração. Type checkers
(e o seu editor) acusam o problema antes mesmo de rodar o código — por exemplo, uma string numa opção que espera booleano.